# 13 — Bures Distance

Close Module 02: lift the Fisher–Rao distance to density matrices, and complete the classical-to-quantum information dictionary.

**Prerequisites:** `02/10_umegaki_relative_entropy`, `02/11_quantum_mutual_information`, `02/12_negative_conditional_entropy`, and `02/08_fisher_rao_geodesics`.

**What you'll be able to do**
- Compute the Bures distance $d_B(\rho, \sigma) = \sqrt{2(1 - F_U)}$ between density matrices.
- See it reduce, for diagonal states, to $\sqrt{2}$ times the Hellinger distance, which is built from the Bhattacharyya coefficient.
- Read the four information rows this synthesis earns, and the whole information block through the dictionary's link — Module 02, the spine, done.

You have lifted entropy, KL, and mutual information to the quantum world. One geometric piece remains: a *distance* between states. The Bures distance is the quantum lift of the Fisher–Rao distance you drew on the simplex in `02/08` — and with it, the whole information spine of the course is complete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import bures_distance
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS

np.random.seed(42)
viz.use_course_style()

## 1. Bures distance — the quantum lift of Fisher–Rao

The Fisher–Rao distance of `02/08` becomes the **Bures distance** in the quantum setting:

$$ d_B(\rho, \sigma) = \sqrt{2\,(1 - F_U(\rho, \sigma))}, \qquad F_U = \operatorname{tr}\sqrt{\sqrt{\rho}\,\sigma\sqrt{\rho}}\ \ (\text{Uhlmann fidelity}). $$

`01/14` called the square $F = F_U^2$ the fidelity; the Bures distance is built on the root $F_U$.

For *diagonal* states $\rho = \operatorname{diag}(p)$, $\sigma = \operatorname{diag}(q)$, the Uhlmann fidelity is $F_U = \sum_i \sqrt{p_i q_i}$ — the **Bhattacharyya coefficient** $B$ of `01/05`, whose square `01/14` found to be the fidelity of commuting states — and Bures becomes $\sqrt{2}$ times the Hellinger distance $\mathrm{Hel}(p, q) = \sqrt{1 - \sum_i \sqrt{p_i q_i}}$, the small-angle cousin of Fisher–Rao. Since the Fisher–Rao distance of `02/08` is $d_{FR} = 2\arccos B$, the identity $1 - \cos x = 2\sin^2(x/2)$ turns this into $d_B = 2\sin(d_{FR}/4)$: on commuting states the Bures distance is a chord of the Fisher–Rao arc, close to $d_{FR}/2$ when the states are near. The factor $\sqrt2$ is a matter of normalisation: $\mathrm{Hel}$ tops out at $1$ for distributions that never overlap, and $d_B$ tops out at $\sqrt{2}$. For *non-commuting* states it reaches beyond what the simplex can see.

**One fidelity by hand: $|+\rangle$ against $I/2$.** A pure state is its own square root, $\sqrt{\rho} = |+\rangle\langle+|$, so the sandwich is $\sqrt\rho\,\sigma\sqrt\rho = |+\rangle\langle+|\,\tfrac I2\,|+\rangle\langle+| = \tfrac12\,|+\rangle\langle+|$. Its square root is $\tfrac1{\sqrt2}\,|+\rangle\langle+|$, with trace $F_U = 1/\sqrt2$, and so $d_B = \sqrt{2(1 - 1/\sqrt2)}$. The cell below checks this against the helper, and checks the two commuting-state identities on one pair of coins.

In [ ]:
pairs = {
    "|+> vs |+>": (density_matrix(KET_PLUS), density_matrix(KET_PLUS)),
    "|0> vs |1>": (density_matrix(KET_0), density_matrix(KET_1)),
    "|+> vs |->": (density_matrix(KET_PLUS), density_matrix(KET_MINUS)),
    "|+> vs I/2": (density_matrix(KET_PLUS), maximally_mixed(2)),
    "|0> vs I/2": (density_matrix(KET_0), maximally_mixed(2)),
}
dists = {k: bures_distance(*v) for k, v in pairs.items()}
print(f"by hand: sqrt(2 (1 - 1/sqrt 2)) = {np.sqrt(2 * (1 - 1 / np.sqrt(2))):.4f}")

# Commuting states: d_B = sqrt(2) Hel = 2 sin(d_FR / 4), checked on one pair of coins.
p, q = np.array([0.9, 0.1]), np.array([0.4, 0.6])
bc = np.sum(np.sqrt(p * q))
d_b = bures_distance(np.diag(p).astype(complex), np.diag(q).astype(complex))
print(f"coins (0.9, 0.1) vs (0.4, 0.6): d_B = {d_b:.4f}, sqrt(2) Hel = {np.sqrt(2) * np.sqrt(1 - bc):.4f}, "
      f"2 sin(d_FR/4) = {2 * np.sin(2 * np.arccos(bc) / 4):.4f}")

# Nearby diagonal states: d_B^2 / dtheta^2 -> Fisher / 4.
theta, h = 0.3, 1e-4
d_near = bures_distance(np.diag([theta, 1 - theta]).astype(complex), np.diag([theta + h, 1 - theta - h]).astype(complex))
print(f"theta = {theta}: d_B^2 / h^2 = {d_near**2 / h**2:.4f}, Fisher / 4 = {1 / (4 * theta * (1 - theta)):.4f}")
print()
for k, d in dists.items():
    print(f"d_B({k:<12s}) = {d:.4f}")

names = list(dists)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(names, [dists[n] for n in names], color=COLORS["quantum"], alpha=0.9)
ax.axhline(np.sqrt(2.0), color=COLORS["highlight"], linestyle="--", label="max = sqrt(2) (orthogonal)")
ax.set_ylabel("Bures distance")
ax.set_title("Orthogonal states reach sqrt(2); |+> vs I/2 is > 0 despite equal Z-diagonals", pad=12)
ax.legend()
plt.show()

**Read the figure and output.** Identical states sit at $0$; orthogonal pure states ($|0\rangle$ vs $|1\rangle$, $|+\rangle$ vs $|-\rangle$) reach the maximum $\sqrt{2} \approx 1.414$. And $|+\rangle$ vs $I/2$ — which share the *same* $Z$-diagonal — is strictly positive: Bures **sees the coherence**, exactly as the Umegaki relative entropy did in `02/10`. The hand value $0.7654$ for this pair is the helper's to the fourth decimal, and the coins check both commuting-state identities.

For two nearby states $\rho_\theta$ and $\rho_{\theta + d\theta}$ the squared distance becomes $d_B^2 = \tfrac14\,F_Q(\theta)\,d\theta^2$, where $F_Q = \mathrm{tr}(\rho_\theta L^2)$ is the *quantum Fisher information* and the symmetric logarithmic derivative $L$ solves $\partial_\theta\rho = \tfrac12(\rho L + L\rho)$ (Braunstein & Caves 1994). On diagonal states $L = \mathrm{diag}(\partial_\theta p_i / p_i)$, $F_Q$ is the classical Fisher information of `02/07`, and the last line of the output checks the factor $\tfrac14$ at $\theta = 0.3$ ($1.1904$ from a step $h = 10^{-4}$ against $1.1905$, checked on this example). Among the monotone metrics that Petz (1996) characterised, this one is the smallest (Petz & Sudár 1996), and it sets the ultimate precision of quantum estimation.

## 2. Dictionary — the information row completed

Module 02 fills in the entire information row of the classical-to-quantum dictionary. Three of the four rows below entered it earlier with their classical side, in `02/02`, `02/04` and `02/07` (the Fisher row came back once more in `02/09`); you have since built their quantum side, in `02/10`, `02/11` and this notebook, so this synthesis earns them again. The fourth, on conditional entropy, enters here: you met its quantum side in `02/12`, and the dictionary records it now that both columns are in your hands.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| relative entropy (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ | Umegaki relative entropy $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/10`) |
| mutual information $I(X;Y) = D(p_{XY}\,\|\,p_X \otimes p_Y) \le \min(H(X), H(Y))$ | quantum mutual information $I(A{:}B) = S(A) + S(B) - S(AB)$, up to twice the classical bound (`02/11`) |
| Fisher information metric on the simplex, $ds^2 = \sum_i dp_i^2/p_i$ | quantum Fisher information: a family of monotone metrics, the smallest being the Bures metric (`02/13`) |
| conditional entropy $H(X\lvert Y) \ge 0$ | quantum conditional entropy $S(A\lvert B)$ — can be negative |

## Your turn

1. **Bures along a Bloch rotation.** For $|\psi(\theta)\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$, tabulate $d_B(|\psi(0)\rangle, |\psi(\theta)\rangle)$ for $\theta \in \{0, \pi/4, \pi/2, \pi\}$ and confirm it equals $2\sin(\theta/4)$.
2. **Diagonal → Hellinger.** For two *diagonal* states $\operatorname{diag}(p)$, $\operatorname{diag}(q)$, confirm $d_B$ reduces to $\sqrt{2(1 - \sum_i \sqrt{p_i q_i})}$, which is $\sqrt{2}$ times the Hellinger distance $\sqrt{1 - \sum_i \sqrt{p_i q_i}}$, the small-angle cousin of the Fisher–Rao distance from `02/08`.
3. **Coherence again.** Confirm $d_B(|+\rangle, I/2) > 0$ even though their $Z$-diagonals match — the same coherence the Umegaki KL saw in `02/10`.

## What you built

- You computed the Bures distance between density matrices and saw orthogonal states reach $\sqrt{2}$.
- You connected it back to the geometry arc: Bures is the quantum lift of the Fisher–Rao distance, reducing to $\sqrt{2}$ times Hellinger for diagonal states.
- You computed one Uhlmann fidelity by hand and met the quantum Fisher information, with its factor $\tfrac14$ and its symmetric logarithmic derivative.
- You completed the information dictionary — and with it, **Module 02, the spine of the course, is done.**

That is a major milestone. You now hold the full information toolbox in both worlds: entropy, divergence, mutual information, conditional entropy, and the metric geometry of distributions and states. Everything the rest of the course needs about *information* is now in your hands. Wonderful work.

## What's next

With information and geometry complete, Module 03 builds **classical optimal transport** in earnest — the "horizontal", mass-moving geometry we glimpsed in `02/09`. It opens with the Monge and Kantorovich problems.

## References

- A. Uhlmann, "The 'transition probability' in the state space of a *-algebra", *Reports on Mathematical Physics* 9, 273–279 (1976). DOI:10.1016/0034-4877(76)90060-4
- M. M. Wilde, *Quantum Information Theory*, ch. 9 & 12, Cambridge University Press (2017).
- S. L. Braunstein & C. M. Caves, "Statistical distance and the geometry of quantum states", *Physical Review Letters* 72, 3439–3443 (1994). DOI:10.1103/PhysRevLett.72.3439
- D. Petz, "Monotone metrics on matrix spaces", *Linear Algebra and its Applications* 244, 81–96 (1996). DOI:10.1016/0024-3795(94)00211-8
- D. Petz & C. Sudár, "Geometries of quantum states", *Journal of Mathematical Physics* 37, 2662–2673 (1996). DOI:10.1063/1.531535

**Previous:** `notebooks/02_InformationAndGeometry/12_negative_conditional_entropy.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/01_the_monge_problem.ipynb`